# Lưu K/V preparation — đúng Output115 thành Dataset
**Accelerator=None, Internet ON, Save & Run All. Không chạy lại chuẩn bị/model.**
Tải đúng version115 của `ducanh2403/2-vi-cai-kia-qua-nhieu`, kiểm full SHA cả8file
gồm inputs.h5, giữ nguyên byte rồi tạo Dataset Private mới. Không lấy latest.
Không cần Add Input. Cần khoảng6GiB trống cho bản tải và bản sao.
Bật Kaggle Secret KAGGLE_API_TOKEN; nếu secret tên codex, đổi SECRET_NAME ở cell đầu.
Không dán token vào code/chat. OWNER phải là tài khoản có quyền xem source115 và tạo Dataset.
Dùng giấy phép Other để giữ các điều khoản nguồn; không tự gán CC0 cho cache.


In [ ]:
from pathlib import Path
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/115'
OWNER='ducanh2403'
SLUG='relation-kv-preparation-v115'
TITLE='Relation KV preparation v115 dataset'
SECRET_NAME='KAGGLE_API_TOKEN'
DESTINATION=Path('/kaggle/working/relation_kv_dataset')


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/115':raise ValueError('This notebook pins115; do not substitute latest or another version.')
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('TITLE/SLUG must have6–50characters')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError(f'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)
VERSION_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/115', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v115', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v115; no network call.', flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
import hashlib,json
INVENTORY={'schema': 'relation_kv_transport_v1', 'kernel_version': 'ducanh2403/2-vi-cai-kia-qua-nhieu/115', 'source_review_zip_sha256': '5c133973195263ba2e34514379875a79597f1bdfcac03f7cf44d74d0961482fa', 'files': {'auxiliary_facts.json': '5b8cbf537fcd3b333d72be21cdcf8883b432dc53f77375ea5751ff5d386d8bd0', 'detector_boxes.json': '9aa664f987986471b937b240341f47e7acbcd64b1bbc6f4cba8ae3d29f3f1a56', 'inputs.h5': '960e1995c6e7695597d95f9535f536576a5c74a03c6c26386da4f7c5b2a4da89', 'metadata.json': '05345ce2ac719a6c7fd70de618c44a3db84d7e31e94a936704fa0370fcbb6742', 'protocol.json': '153e069b1aa3051ee8f805998aac9894bc1ebd4842e962d3ee84a15ff67f45fe', 'source_receipts.json': '0a78ea6b5a40c1ebc64158750363bf7d2ad42cb4bb6360350ba2771e6bba0219', 'statistics.json': '7e981ca71028beaf7f44880553c1a101ddab9a1b0e28ce26f1cfb7bca73969d4', 'cpu_preparation.json': 'bcfeb4f263f3860374e3d255f66b0ed8912816413706f3444b3091aa4d8af1a8'}}
SOURCE_CODE='"""Verify/copy original CPU preparation bytes; no API, HDF5 reader, or model."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\n\nNAMES = {\'auxiliary_facts.json\',\'cpu_preparation.json\',\'detector_boxes.json\',\n         \'inputs.h5\',\'metadata.json\',\'protocol.json\',\'source_receipts.json\',\'statistics.json\'}\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda:stream.read(8*1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef package(source, destination, inventory):\n    expected = inventory[\'files\']\n    if inventory[\'schema\'] != \'relation_kv_transport_v1\' or set(expected) != NAMES:\n        raise ValueError(\'Wrong inventory\')\n    if any(not re.fullmatch(\'[0-9a-f]{64}\', h) for h in expected.values()):\n        raise ValueError(\'Invalid pinned hashes\')\n    source, destination = Path(source).resolve(), Path(destination).resolve()\n    if source == destination or source in destination.parents or destination in source.parents:\n        raise ValueError(\'Use separate non-nested folders\')\n    for name, expected_sha in expected.items():\n        path = source/name\n        if path.is_symlink() or not path.is_file() or sha(path) != expected_sha:\n            raise ValueError(\'Missing/changed source: \'+name)\n        print(\'Verified source:\',name,flush=True)\n    receipt = json.loads((source/\'cpu_preparation.json\').read_bytes())\n    if receipt[\'status\'] != \'READY_FOR_MODEL_PREFLIGHT\' or receipt[\'rows\'] != 5472:\n        raise ValueError(\'Wrong completed preparation\')\n    if any(receipt[k] for k in (\'neural_forward\',\'feature_extraction\',\'checkpoint_loaded\')):\n        raise ValueError(\'Unexpected model execution\')\n    if receipt[\'payload_sha256\'] != {n:h for n,h in expected.items() if n != \'cpu_preparation.json\'}:\n        raise ValueError(\'Receipt payload manifest differs\')\n    destination.mkdir(parents=True,exist_ok=True)\n    if any(p.is_symlink() or p.name not in NAMES|{\'transport_receipt.json\'} for p in destination.iterdir()):\n        raise ValueError(\'Unrelated or partial destination preserved\')\n    needed = sum((source/n).stat().st_size for n in expected if not (destination/n).exists())\n    if shutil.disk_usage(destination).free < needed+256*1024**2:\n        raise RuntimeError(\'Insufficient copy space\')\n    for name, expected_sha in expected.items():\n        target = destination/name\n        if target.exists():\n            if not target.is_file() or sha(target) != expected_sha:\n                raise ValueError(\'Changed destination preserved: \'+name)\n            continue\n        partial = destination/(name+\'.partial\')\n        shutil.copyfile(source/name,partial)\n        if sha(partial) != expected_sha:\n            raise ValueError(\'Copy differs: \'+name)\n        partial.replace(target)\n    report = dict(status=\'DATASET_FOLDER_READY\',schema=inventory[\'schema\'],files=expected,\n                  source_kernel_version=inventory[\'kernel_version\'],payload_files=len(NAMES),\n                  source_review_zip_sha256=inventory[\'source_review_zip_sha256\'],\n                  full_payload_hashes_verified=True,model_executed=False,\n                  note=\'Original eight files unchanged at dataset root. No HDF5/model execution.\')\n    (destination/\'transport_receipt.json\').write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    return report\n'
assert hashlib.sha256(SOURCE_CODE.encode()).hexdigest()=='802700ea2ad32949ed445e35d2717e900472db8ed1a6c2595b81995856a8b498'
exec(compile(SOURCE_CODE,'package_kv_dataset.py','exec'))


In [ ]:
import tempfile,shutil
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<6*1024**3:
    raise RuntimeError('Need about6GiB free for download+copy; preserve existing outputs.')
DOWNLOAD=Path(tempfile.mkdtemp(prefix='relation-kv-v115-',dir='/tmp'))
pattern=r'(^|/)(?:'+'|'.join(re.escape(n) for n in INVENTORY['files'])+r')$'
print('Download EXACT source:',KERNEL_VERSION,flush=True)
run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),'--file-pattern',pattern])
matches=[p.parent for p in DOWNLOAD.rglob('cpu_preparation.json') if sha(p)==INVENTORY['files']['cpu_preparation.json']]
if len(matches)!=1:raise RuntimeError(f'Output115 returned {len(matches)} matching receipts; STOP, no latest fallback.')
report=package(matches[0],DESTINATION,INVENTORY)
print('Eight original files verified and copied:',DESTINATION,flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private research preparation from exact Output115. Existing COCO CLIP/legacy feature tensors, predicted boxes and VSR observed relation metadata. Upstream data/model terms retained. No original pixels, new encoder, trained model or caption-quality claim. Original source/transport receipts preserved.')
if metadata_path.exists():raise RuntimeError('Preserve existing metadata; use a fresh destination.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('API accepted creation; processing may be pending. Do not repeatedly create.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','status',f'{OWNER}/{SLUG}'])
except RuntimeError:print('Status not confirmed; check dataset page before retrying.',flush=True)


Dataset có8file gốc và transport_receipt.json ngay thư mục gốc; không chứa repo/log/sourcefiles.
Nếu giữ mặc định, path mount dự kiến:
`/kaggle/input/datasets/ducanh2403/relation-kv-preparation-v115`.
Gửi path thực tế sau khi attach. Chia sẻ Private với Ngân để chạy nhánh thứ hai.
Giữ Output115 đến khi Dataset xử lý xong. Không cần upload checkpoint mới.
Notebook chỉ được kiểm metadata/static/mock; Codex chưa gọi API hoặc chạy trên Kaggle.
